In [3]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime, gc, json
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
from pyspark.mllib.evaluation import RankingMetrics
from pyspark.sql.types import StringType

# Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/"
CHECKPOINT_DIR = BASE_PATH + "spark_checkpoints/"

os.makedirs(OUTPUT_DIR + "models/", exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Khởi tạo Spark
spark = SparkSession.builder \
    .appName("HM_8Week_Pipeline") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
print("✅ Spark Ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Spark Ready!


In [4]:
# 1. Đọc dữ liệu
df = spark.read.parquet(INPUT_FILE)
max_date = df.select(F.max("t_dat_date")).collect()[0][0]

# 2. XÁC ĐỊNH CÁC MỐC THỜI GIAN (8 TUẦN)
test_start_date = max_date - datetime.timedelta(days=7)      # Tuần 8 (Test)
val_start_date = test_start_date - datetime.timedelta(days=7) # Tuần 7 (Validation)
train_start_date = val_start_date - datetime.timedelta(weeks=6) # Tuần 1-6 (Train)

# 3. CHIA DỮ LIỆU
# Tầng 1: Train Retrieval (W1 -> W6)
train_retrieval = df.filter((F.col("t_dat_date") >= train_start_date) & (F.col("t_dat_date") < val_start_date))

# Tầng 2: Validation Ranking (W7) - Dùng để gán nhãn train XGBoost
val_ranking = df.filter((F.col("t_dat_date") >= val_start_date) & (F.col("t_dat_date") < test_start_date))

# Tầng 3: Test Final (W8) - Dùng để đánh giá cuối cùng hoặc Web Demo
test_final = df.filter(F.col("t_dat_date") >= test_start_date)

# 4. Indexing cố định trên tập Train (W1-6)
u_model = StringIndexer(inputCol="customer_id", outputCol="user_idx").setHandleInvalid("skip").fit(train_retrieval)
i_model = StringIndexer(inputCol="article_id", outputCol="item_idx").setHandleInvalid("skip").fit(train_retrieval)

# Chuẩn bị Ground Truth cho Validation (Tuần 7)
val_gt_indexed = i_model.transform(u_model.transform(val_ranking)) \
    .groupBy("user_idx") \
    .agg(F.collect_list("item_idx").alias("actual_item_idxs")) \
    .cache()

print(f"📅 Train (W1-6): {train_start_date} -> {val_start_date}")
print(f"📅 Val   (W7):   {val_start_date} -> {test_start_date}")
print(f"📅 Test  (W8):   {test_start_date} -> {max_date}")

📅 Train (W1-6): 2020-07-28 -> 2020-09-08
📅 Val   (W7):   2020-09-08 -> 2020-09-15
📅 Test  (W8):   2020-09-15 -> 2020-09-22


In [5]:
BEST_DECAY = 0.1
BEST_RANK = 80
BEST_ALPHA = 40

print("🚀 Đang huấn luyện ALS trên dữ liệu 6 tuần đầu...")

# Tính Ratings với Time Decay (Tính đến ngày bắt đầu tuần Validation)
ratings = train_retrieval.withColumn("days_diff", F.datediff(F.lit(val_start_date), F.col("t_dat_date"))) \
                         .withColumn("weight", F.exp(-BEST_DECAY * F.col("days_diff"))) \
                         .groupBy("customer_id", "article_id") \
                         .agg(F.sum("weight").alias("rating"))

train_indexed = i_model.transform(u_model.transform(ratings)).repartition(32).checkpoint()

als_model = ALS(maxIter=15, rank=BEST_RANK, regParam=0.1, alpha=BEST_ALPHA,
                checkpointInterval=10, userCol="user_idx", itemCol="item_idx", ratingCol="rating",
                implicitPrefs=True, coldStartStrategy="drop", nonnegative=True).fit(train_indexed)

print("✅ Huấn luyện ALS hoàn tất!")

🚀 Đang huấn luyện ALS trên dữ liệu 6 tuần đầu...
✅ Huấn luyện ALS hoàn tất!


In [6]:
print("🎯 Đang tạo 100 ứng viên cho tuần Validation (Tuần 7)...")

# Lấy top 100 ứng viên từ ALS
val_recs = als_model.recommendForUserSubset(val_gt_indexed.select("user_idx"), 100).cache()

# Tính toán các chỉ số cho báo cáo BTL
eval_rdd = val_recs.join(val_gt_indexed, "user_idx") \
    .select(F.col("recommendations.item_idx").alias("p"), "actual_item_idxs") \
    .rdd.map(lambda r: (list(r[0]), list(r[1]))).cache()

map12 = RankingMetrics(eval_rdd.map(lambda x: (x[0][:12], x[1]))).meanAveragePrecision
print(f"🏆 MAP@12 (Pure ALS trên Tuần 7): {map12:.6f}")

🎯 Đang tạo 100 ứng viên cho tuần Validation (Tuần 7)...


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


🏆 MAP@12 (Pure ALS trên Tuần 7): 0.015263


In [7]:
print("💾 Đang lưu trữ dữ liệu...")

# 1. Lưu Model & Indexers
als_model.write().overwrite().save(OUTPUT_DIR + "models/als_best_model")
u_model.write().overwrite().save(OUTPUT_DIR + "models/user_indexer")
i_model.write().overwrite().save(OUTPUT_DIR + "models/item_indexer")

# 2. Lưu ứng viên tuần 7 (Để làm dữ liệu TRAIN cho Reranking)
val_recs.write.mode("overwrite").parquet(OUTPUT_DIR + "candidates/als_top100_val_W7.parquet")

# 3. Lưu thêm thông tin thực tế tuần 7 (Để gán nhãn Label 0/1)
val_gt_indexed.write.mode("overwrite").parquet(OUTPUT_DIR + "candidates/actual_labels_W7.parquet")

print(f"✅ Đã lưu mọi thứ phục vụ bước Reranking tại: {OUTPUT_DIR}")

💾 Đang lưu trữ dữ liệu...
✅ Đã lưu mọi thứ phục vụ bước Reranking tại: /content/drive/MyDrive/HM-DATA/outputs/
